# Build and validate NEX-Laminate v1.0

This notebook regenerates the dataset from its declared material card and
manufacturing rules. Numerical outputs are `F0_CLT`; they are not represented
as experimental observations.

## Governing equations

For ply interfaces \(z_k\) and transformed reduced stiffness
\(\bar{\mathbf Q}^{(k)}\),

\[
\mathbf A=\sum_k\bar{\mathbf Q}^{(k)}(z_k-z_{k-1}),\quad
\mathbf B=\frac12\sum_k\bar{\mathbf Q}^{(k)}(z_k^2-z_{k-1}^2),\quad
\mathbf D=\frac13\sum_k\bar{\mathbf Q}^{(k)}(z_k^3-z_{k-1}^3).
\]

With \(\mathbf a=\mathbf A^{-1}\),
\(E_x=(a_{11}h)^{-1}\), \(E_y=(a_{22}h)^{-1}\),
\(G_{xy}=(a_{66}h)^{-1}\), and
\(\nu_{xy}=-a_{12}/a_{11}\).

The source code also evaluates sign-dependent Tsai-Hill and maximum-stress
first-ply capacities at both surfaces of every ply.

In [ ]:
from pathlib import Path
import json, subprocess, sys

HERE = Path.cwd().resolve()
ROOT = HERE.parent if HERE.name == "notebooks" else HERE
GENERATOR = ROOT / "src" / "generate_inverse_dataset.py"
VALIDATOR = ROOT / "src" / "validate_inverse_dataset.py"
assert GENERATOR.exists() and VALIDATOR.exists(), ROOT
print("Dataset root:", ROOT)

In [ ]:
subprocess.run(
    [sys.executable, str(GENERATOR), "--output", str(ROOT)],
    check=True,
)

In [ ]:
subprocess.run(
    [sys.executable, str(VALIDATOR), "--dataset", str(ROOT)],
    check=True,
)
qa = json.loads((ROOT / "reports" / "qa_report.json").read_text())
qa

In [ ]:
import csv

with (ROOT / "data" / "numerical_clt_v1_sample.csv").open(newline="") as f:
    rows = list(csv.DictReader(f))
print("Sample rows:", len(rows))
for row in rows[:3]:
    print(row["record_id"], row["layup_deg"], row["Ex_GPa"], row["split"])

## Publication gate

Do not describe this release as component-level validation. Add the paired
physical observations defined in `protocols/` before making experimental
accuracy claims for manufactured laminates or landing gear.